# 05: Comprehensive Validation Report - Tier-2 and Tier-3

Assembles per-vendor validation tables from the **measured** JSON exports in
`reports/comparison/<vendor>/`. No circuit is re-simulated here and no number
is invented: Tier-2 endpoint/full-sweep and DPT figures and Tier-3 static-fit,
DPT and runtime figures are read verbatim from those JSONs. Missing crossings
remain `not available` (IRFS Tier-3 `td_on`/`td_off`/`e_on`/`e_off`).

In [1]:
import sys, os, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd

VENDORS = ('irfs7730pbf', 'csd16412q5a', 'epc2216', 'cpm2-1200')
CMP = REPO_ROOT / 'reports' / 'comparison'
OUT_HTML = REPO_ROOT / 'reports' / 'validation_report.html'

In [2]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def disp(v, pct=False):
    if v is None:
        return 'not available'
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v)
    if not np.isfinite(f):
        return 'not available'
    return f'{f:+.2f}%' if pct else f'{f:.4g}'


CAVEAT = ('Manufacturer datasheets were checked against extracted currents at the '
          'rated gate drive: IRFS7730PBF 246 A continuous / 984 A pulsed vs 1671 A '
          'extracted at Vgs=10 V; CSD16412Q5A 52 A / 91 A vs 305 A; EPC2216 3.4 A / '
          '28 A vs 61.7 A at Vgs=6 V; CPM2-1200 81 A / 200 A vs 342 A at Vgs=20 V. '
          'These excesses are properties of the vendor subcircuits (VM-04/VM-06). '
          'Matching the vendor golden is not the same as matching the datasheet; '
          'fidelity claims against the vendor must not be read as physical device '
          'ratings.')

In [3]:
sections, summary = [], []
for vendor in VENDORS:
    base = CMP / vendor
    ep = load_json(base / 'endpoint_comparison.json')
    fs = load_json(base / 'full_sweep_comparison.json')
    d2 = load_json(base / 'dpt_metrics.json')
    d3 = load_json(base / 'tier3_validation.json')

    t2_end = pd.DataFrame([{
        'Vgs [V]': r['vgs_v'], 'golden [A]': disp(r['golden_a']),
        'Tier-2 [A]': disp(r['tier2_a']),
        'rel err': disp(r['rel_err_pct'], pct=True)} for r in ep])
    t2_sweep = pd.DataFrame([{
        'curve': r['curve'], 'points': r['points'],
        'rel RMSE': disp(r['rel_rmse_pct'], pct=True),
        'peak RMSE': disp(r['peak_rmse_pct'], pct=True),
        'max rel err': disp(r['max_rel_pct'], pct=True),
        'R^2': disp(r['r_squared']), 'pass': r['pass']} for r in fs])
    fit = d3['function_fit']
    t3_static = pd.DataFrame([
        {'quantity': 'Tier-3 static fit rel RMSE', 'value': disp(fit['static_curve_rmse_rel'] * 100, pct=True)},
        {'quantity': 'Tier-3 emitted static rel RMSE', 'value': disp(fit['emitted_static_curve_rmse_rel'] * 100, pct=True)},
        {'quantity': 'Tier-3 held-out Id rel RMSE', 'value': disp(fit['heldout_id_rmse_rel'] * 100, pct=True)},
        {'quantity': 'Tier-3 held-out Qg rel RMSE', 'value': disp(fit['heldout_qg_rmse_rel'] * 100, pct=True)},
        {'quantity': 'static samples', 'value': fit['static_samples']},
    ])
    t3_dpt = pd.DataFrame([{
        'metric': k, 'golden': disp(c['golden']),
        'Tier-2': disp(c['tier2']), 'Tier-3': disp(c['tier3']),
        'T2 err': disp(c['tier2_rel_err_pct'], pct=True),
        'T3 err': disp(c['tier3_rel_err_pct'], pct=True)}
        for k, c in d3['dpt_comparison'].items()])
    dr, wf, rt = d3['dpt_run'], d3['dpt_run']['waveform'], d3['runtime']
    t3_run = pd.DataFrame([
        {'quantity': 'DPT completed', 'value': dr['completed']},
        {'quantity': 'golden trace points', 'value': dr['golden_trace_points']},
        {'quantity': 'Tier-3 trace points', 'value': dr['trace_points']},
        {'quantity': 'Tier-3 iterations', 'value': dr['iterations']},
        {'quantity': 'DPT Vds rel RMSE', 'value': disp(wf['vds_rel_rmse_pct'], pct=True)},
        {'quantity': 'DPT Id rel RMSE', 'value': disp(wf['id_rel_rmse_pct'], pct=True)},
        {'quantity': 'golden solver elapsed [s]', 'value': rt['golden']['solver_elapsed_s']},
        {'quantity': 'Tier-3 solver elapsed [s]', 'value': rt['tier3']['solver_elapsed_s']},
        {'quantity': 'solver time ratio (measured)', 'value': disp(rt['solver_speedup'])},
        {'quantity': 'Tier-2 golden failure', 'value': d2.get('golden_failure') or 'none'},
    ])

    print(f'\n===== {vendor} : Tier-2 endpoint comparison ====='); display(t2_end)
    print(f'----- {vendor} : Tier-2 full-sweep fit -----'); display(t2_sweep)
    print(f'----- {vendor} : Tier-3 static fit -----'); display(t3_static)
    print(f'----- {vendor} : DPT timing / energy (golden vs T2 vs T3) -----'); display(t3_dpt)
    print(f'----- {vendor} : Tier-3 DPT run / runtime -----'); display(t3_run)

    sections.append(f'<h2>{vendor}</h2>'
                    + '<h3>Tier-2 endpoint comparison</h3>' + t2_end.to_html(index=False)
                    + '<h3>Tier-2 full-sweep fit</h3>' + t2_sweep.to_html(index=False)
                    + '<h3>Tier-3 static fit</h3>' + t3_static.to_html(index=False)
                    + '<h3>DPT timing / energy</h3>' + t3_dpt.to_html(index=False)
                    + '<h3>Tier-3 DPT run / runtime</h3>' + t3_run.to_html(index=False))
    summary.append({
        'vendor': vendor,
        'T2 static mean rel RMSE': disp(np.mean([r['rel_rmse_pct'] for r in fs]), pct=True),
        'T3 static fit rel RMSE': disp(fit['static_curve_rmse_rel'] * 100, pct=True),
        'T3 emitted rel RMSE': disp(fit['emitted_static_curve_rmse_rel'] * 100, pct=True),
        'DPT Vds RMSE': disp(wf['vds_rel_rmse_pct'], pct=True),
        'DPT Id RMSE': disp(wf['id_rel_rmse_pct'], pct=True),
        'solver time ratio': disp(rt['solver_speedup']),
    })


===== irfs7730pbf : Tier-2 endpoint comparison =====


,Vgs [V],golden [A],Tier-2 [A],rel err
0,3.0,2.021e-07,2.021e-07,+0.00%
1,4.5,7.159,6.875,+3.96%
2,6.0,340.1,327.5,+3.72%
3,8.0,975.1,940.1,+3.59%
4,10.0,1671,1620,+3.08%


----- irfs7730pbf : Tier-2 full-sweep fit -----


,curve,points,rel RMSE,peak RMSE,max rel err,R^2,pass
0,Vgs = 3 V,41,+10.13%,+10.13%,+14.76%,1,False
1,Vgs = 4.5 V,41,+13.85%,+13.85%,+49.72%,0.2562,False
2,Vgs = 6 V,41,+4.64%,+4.64%,+9.41%,0.9497,True
3,Vgs = 8 V,41,+3.07%,+3.07%,+6.87%,0.987,True
4,Vgs = 10 V,41,+3.65%,+3.65%,+7.05%,0.9857,True


----- irfs7730pbf : Tier-3 static fit -----


,quantity,value
0,Tier-3 static fit rel RMSE,+4.24%
1,Tier-3 emitted static rel RMSE,+4.10%
2,Tier-3 held-out Id rel RMSE,+3327.16%
3,Tier-3 held-out Qg rel RMSE,+8.47%
4,static samples,505


... (30 outputs left out)

----- cpm2-1200 : DPT timing / energy (golden vs T2 vs T3) -----


,metric,golden,Tier-2,Tier-3,T2 err,T3 err
0,td_on_ns,10.25,11.78,8.641,+14.90%,-15.72%
1,tr_ns,4.703,4.123,0.814,-12.33%,-82.69%
2,td_off_ns,14.03,18.24,7.764,+29.98%,-44.68%
3,tf_ns,4.139,8.789,2.53,+112.33%,-38.87%
4,dv_dt_max_v_per_ns,373.7,374.9,1170,+0.33%,+213.10%
5,di_dt_max_a_per_ns,0.5957,0.3811,0.4946,-36.03%,-16.98%
6,e_on_uj,5.619,6.109,4.465,+8.73%,-20.53%
7,e_off_uj,56.7,57.04,55.43,+0.60%,-2.25%
8,i_peak_a,26.82,26.73,26.65,-0.34%,-0.65%


----- cpm2-1200 : Tier-3 DPT run / runtime -----


,quantity,value
0,DPT completed,True
1,golden trace points,7154
2,Tier-3 trace points,31436
3,Tier-3 iterations,189005
4,DPT Vds rel RMSE,+4.21%
5,DPT Id rel RMSE,+0.35%
6,golden solver elapsed [s],0.532
7,Tier-3 solver elapsed [s],5.185
8,solver time ratio (measured),0.1026
9,Tier-2 golden failure,none


## Sources and caveats

All values are read from `reports/comparison/<vendor>/{endpoint_comparison,
full_sweep_comparison,dpt_metrics,tier3_validation}.json`. The `T2 static mean
rel RMSE` is the arithmetic mean of the measured per-curve `rel_rmse_pct`
values in `full_sweep_comparison.json`. All measured solver time ratios are
below 1.0 (a slowdown); no speedup is claimed.

Manufacturer datasheets were checked against extracted currents at the rated
gate drive: IRFS7730PBF 246 A continuous / 984 A pulsed vs 1671 A extracted at
Vgs=10 V; CSD16412Q5A 52 A / 91 A vs 305 A; EPC2216 3.4 A / 28 A vs 61.7 A at
Vgs=6 V; CPM2-1200 81 A / 200 A vs 342 A at Vgs=20 V. These excesses are
properties of the vendor subcircuits (VM-04/VM-06). Matching the vendor golden
is not the same as matching the datasheet; fidelity claims against the vendor
must not be read as physical device ratings.

In [4]:
print('Consolidated measured validation summary')
display(pd.DataFrame(summary).set_index('vendor'))
html = ('<html><head><meta charset="utf-8"><title>PieceSpice Validation Report</title></head><body>'
        '<h1>PieceSpice Comprehensive Validation - measured JSON values</h1>'
        + pd.DataFrame(summary).to_html(index=False)
        + '<p>' + CAVEAT + '</p>'
        + ''.join(sections) + '</body></html>')
OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
OUT_HTML.write_text(html, encoding='utf-8')
print(f'HTML report: {OUT_HTML.resolve()}')

Consolidated measured validation summary


,T2 static mean rel RMSE,T3 static fit rel RMSE,T3 emitted rel RMSE,DPT Vds RMSE,DPT Id RMSE,solver time ratio
vendor,,,,,,
irfs7730pbf,+7.07%,+4.24%,+4.10%,+39.59%,+13.09%,0.2156
csd16412q5a,+8.49%,+1.54%,+2.15%,+2.38%,+0.42%,0.06671
epc2216,+2.04%,+2.37%,+7.12%,+5.29%,+0.76%,0.229
cpm2-1200,+3.43%,+0.61%,+0.97%,+4.21%,+0.35%,0.1026


HTML report: C:\Users\GNKRISHNAN\Documents\GitHub\PieceSpice\reports\validation_report.html
